In [1]:
from typing import Any, Callable, Tuple, Dict
import functools
import inspect
import asyncio
import nest_asyncio
import time
import ast
import sys
import traceback


# This library patches asyncio to allow nested event loops
nest_asyncio.apply()

from functools import wraps
from copy import copy

def inject_context(func, __local_context):
    @wraps(func)
    def wrapped(*args, **kwargs):
        if "__local_context" in kwargs:
            raise RuntimeError("__local_context is already set")

        # defensive copy per invocation
        kwargs["__local_context"] = copy(__local_context)

        return func(*args, **kwargs)
    return wrapped

def tool(f):
    """Marks a function as a tool for LLM agent."""

    @functools.wraps(f)
    def wrapper(*args, **kwargs):
        # update globals with local context
        result = None
        if inspect.iscoroutinefunction(f):
            # If f is async, run it using the event loop
            result = asyncio.get_running_loop().run_until_complete(f(*args, **kwargs))
        else:
            # If f is sync, just call it
            result = f(*args, **kwargs)
    return wrapper

In [2]:
@tool
async def async_wait_function(name, __local_context=None):
    print(f"MESSGE FROM ASYNC FUNCT {name}: {__local_context["local_message"]} -- {__local_context["message"]}")
    await asyncio.sleep(3)
    print(f"END MESSGE FROM ASYNC FUNCT {name}: {__local_context["local_message"]} -- {__local_context["message"]}")

In [3]:
async def run_async(code, context, tools):
    global_context = globals()
    for name, funct in tools.items():
        global_context[name] = inject_context(funct, context)
        
    tree = ast.parse(code)
    for node in tree.body:
        wrapper = ast.Module(body=[node], type_ignores=[])
        ast.fix_missing_locations(wrapper)
        
        try:
            code_obj = compile(wrapper, filename="<string>", mode="exec")
            exec(code_obj, global_context, local_context)
        except Exception as ex:
            print(f"Exception {ex}")

In [4]:
tools = {
    "async_function":async_wait_function
}
local_context = {
    "message": "some_message"
}
code = """
local_message="local message"
async_function("FUNCTION 1")"""

code2 = """
local_message="local message second"
async_function("FUNCTION 2")
local_message="local message second after async"
"""
cor1 = run_async(code, local_context, tools)
cor2 = run_async(code2, local_context, tools)

results = await asyncio.gather(cor1, cor2)

MESSGE FROM ASYNC FUNCT FUNCTION 1: local message -- some_message
MESSGE FROM ASYNC FUNCT FUNCTION 2: local message second -- some_message
END MESSGE FROM ASYNC FUNCT FUNCTION 1: local message -- some_message
END MESSGE FROM ASYNC FUNCT FUNCTION 2: local message second -- some_message
